# House Prices Project

**Dataset**: [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)

**Goal**: Predict the sale price of houses using statistical and machine learning methods.


---
## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add your additional imports as needed
# from sklearn...
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations
import statsmodels.formula.api as smf
# import torch

pd.set_option('display.max_columns', 100)
%matplotlib inline

---
## Load Data

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f'Training set: {train.shape[0]} rows, {train.shape[1]} columns')
print(f'Test set: {test.shape[0]} rows, {test.shape[1]} columns')

train.head()

In [ ]:
# Explore the target variable
print(train['SalePrice'].describe())

fig, axes = plt.subplots(1, 1, figsize=(12, 4))
axes.hist(train['SalePrice'], bins=50, edgecolor='black')
axes.set_title('SalePrice Distribution')
axes.set_xlabel('SalePrice')
plt.tight_layout()
plt.show()

print(f'Skewness: {train["SalePrice"].skew():.3f}')

The data is heavily skewed, what kind of transformation can we apply to make it more normal? 

In [ ]:
# Apply a log transformation so y' = log(price + 1) (+1 for safety/robustness)
# Then we can interpret coefficients as percentage changes instead of absolute price changes

df = train.copy()
df["SalePriceLog"] = np.log1p(df["SalePrice"])

print(df['SalePriceLog'].describe())

fig, axes = plt.subplots(1, 1, figsize=(12, 4))
axes.hist(df['SalePriceLog'], bins=50, edgecolor='black')
axes.set_title('SalePriceLog Distribution')
axes.set_xlabel('SalePriceLog')
plt.tight_layout()
plt.show()

print(f'Skewness: {df["SalePriceLog"].skew():.3f}')

In [ ]:
# Explore missing values
(df.isna().sum()[df.isna().sum() > 0] / len(df)).sort_values(ascending=False) * 100

In [ ]:
# Let's clean up!
# Assume no data about pool -> no pool, and same for misc, alley and fence
df["PoolQC"] = df["PoolQC"].fillna("NA")
df["MiscFeature"] = df["MiscFeature"].fillna("NA")
df["Alley"] = df["Alley"].fillna("NA")
df["Fence"] = df["Fence"].fillna("NA")

# Assume no data about masonry area -> area = 0
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
# If masonry area is 0, veneer type should be none
df.loc[(df["MasVnrArea"] == 0) & (df["MasVnrType"].isna()), "MasVnrType"] = "None"
# For any with masonry area > 0 but veneer type NaN (4 rows) -> fill with most common veneer type
df["MasVnrType"] = df["MasVnrType"].fillna(df.loc[df["MasVnrArea"] > 0, "MasVnrType"].mode()[0])

# No fireplaces -> fireplace quality is NA
df.loc[(df["Fireplaces"] == 0) & (df["FireplaceQu"].isna()), "FireplaceQu"] = "NA"

# Same treatment for garage
df.loc[(df["GarageArea"] == 0) & (df["GarageFinish"].isna()), "GarageFinish"] = "NA"
df.loc[(df["GarageArea"] == 0) & (df["GarageQual"].isna()), "GarageQual"] = "NA"
df.loc[(df["GarageArea"] == 0) & (df["GarageType"].isna()), "GarageType"] = "NA"
df.loc[(df["GarageArea"] == 0) & (df["GarageCond"].isna()), "GarageCond"] = "NA"
# Add a separate column for garage/no garage (maybe we don't need this)
df["HasGarage"] = df["GarageYrBlt"].notna().astype(int)
# Set year built to 0 if no garage
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

# Again for basement: no basement square feet -> no basement
df.loc[(df["TotalBsmtSF"] == 0), ["BsmtFinType1", "BsmtFinType2", "BsmtExposure", "BsmtQual", "BsmtCond"]] = "NA"
# Deal with any leftover missing values
df[["BsmtFinType2", "BsmtExposure"]] = df[["BsmtFinType2", "BsmtExposure"]].fillna("NA")

# Set missing lot frontage to the neighbourhood median
df["LotFrontage"] = df["LotFrontage"].fillna(
    df.groupby("Neighborhood")["LotFrontage"].transform("median")
)

# Fill in electrical with most frequent value
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# Just because 'NA' is confusing, let's rename it to 'None'
df = df.replace("NA", "None")

In [ ]:
# Check everything is clean
(df.isna().sum()[df.isna().sum() > 0] / len(df)).sort_values(ascending=False) * 100

---
## Part 1: Classical Statistical Inference

Apply basic statistical methods to explore the data:
- **Sample mean and variance** of `SalePrice` and key features
- **Confidence intervals** for the mean SalePrice
- **Hypothesis testing** — e.g. is the mean SalePrice significantly different from \$180,000? Is the distribution of the transformed `SalePrice` normal (Shapiro-Wilk)?
- Visualize distributions and support your conclusions with plots

---
## Part 2: ANOVA — Finding Significant Features

Use ANOVA to determine which of the following **10 features** have a statistically significant effect on the transformed SalePrice. 

**Given features (10):**

| # | Feature | Levels | Description |
|---|---|---|---|
| 1 | `OverallQual` | 1–10 | Overall material and finish quality |
| 2 | `ExterQual` | Po, Fa, TA, Gd, Ex | Exterior material quality |
| 3 | `BsmtQual` | None, Po, Fa, TA, Gd, Ex | Basement height quality |
| 4 | `KitchenQual` | Po, Fa, TA, Gd, Ex | Kitchen quality |
| 5 | `FireplaceQu` | None, Po, Fa, TA, Gd, Ex | Fireplace quality |
| 6 | `CentralAir` | N, Y | Central air conditioning |
| 7 | `LotShape` | IR3, IR2, IR1, Reg | General shape of property |
| 8 | `LandSlope` | Sev, Mod, Gtl | Slope of property |
| 9 | `MoSold` | 1–12 | Month sold |
| 10 | `YrSold` | 2006–2010 | Year sold |

**Tasks:**
1. Extract these features into a dataframe and run **one-way ANOVA** on each
2. Identify which features are significant (p < 0.05)
3. Run a **two-way ANOVA** to test for interaction effects between pairs of significant features
4. Use **Tukey HSD** post-hoc tests where appropriate
5. Summarize: which features and interactions are significant?

In [ ]:
features = ["OverallQual", "ExterQual", "BsmtQual", "KitchenQual", "FireplaceQu", "CentralAir", "LotShape", "LandSlope", "MoSold", "YrSold"]

df_for_anova = df[features + ["SalePriceLog"]]
df_for_anova.head()

In [ ]:
results = []

for feature in features:
  groups = [group["SalePriceLog"].values for name, group in df_for_anova.groupby(feature)]
  f_stat, p_value = stats.f_oneway(*groups)

  results.append({
      "feature": feature,
      "f_statistic": f_stat,
      "p_value": p_value,
      "num_groups": len(groups)
  })

anova_df = pd.DataFrame(results)
anova_df = anova_df.sort_values("p_value")
anova_df["significant"] = anova_df["p_value"] < 0.05
anova_df

In [ ]:
significant_features = anova_df[anova_df["significant"]]["feature"].tolist()
print(significant_features)

In [ ]:
results = []

for f1, f2 in combinations(significant_features, 2):
    formula = f"SalePriceLog ~ C({f1}) * C({f2})"
    model = smf.ols(formula, data=df_for_anova).fit()
    anova = sm.stats.anova_lm(model, typ=2)

    results.append({
        "feature1": f1,
        "feature2": f2,
        "p_interaction": anova.loc[f"C({f1}):C({f2})", "PR(>F)"]
    })

anova2_df = pd.DataFrame(results).sort_values("p_interaction")
anova2_df["significant"] = anova2_df["p_interaction"] < 0.05
anova2_df

In [ ]:
# Check frequency of combinations using contingency table
print(pd.crosstab(df_for_anova["OverallQual"], df_for_anova["KitchenQual"]))
print(pd.crosstab(df_for_anova["FireplaceQu"], df_for_anova["LotShape"]))

In [ ]:
for feature in significant_features:
    print("\n" + "="*50)
    print(f"Tukey HSD for {feature}")
    print("="*50)

    tukey = pairwise_tukeyhsd(
        endog=df_for_anova["SalePriceLog"],
        groups=df_for_anova[feature],
        alpha=0.05
    )

    print(tukey)

---
## Part 3: 2^k Factorial Design

Pick k binary (or binarized) factors from the significant features found in Part 2 and apply a factorial design analysis. For example you could binarize ordinal features into High/Low groups and study their joint effects.

**Tasks:**
- Select k factors (e.g. k=2 or k=3) and define High/Low levels
- Compute group means for all 2^k combinations
- Analyze main effects and interaction effects
- Visualize with interaction plots

---
## Part 4: Parametric Regression

Build a regression model using only the **significant ordinal features** identified by ANOVA (Part 2) plus the **2 numerical features**: `GrLivArea` and `TotalBsmtSF`.

**Tasks:**
- Encode ordinal features numerically (map quality levels to integers)
- Fit a linear regression model (OLS)
- Analyze the model: R², coefficient significance, residual plots
- Optionally try regularized regression (Ridge, Lasso) and compare
- Apply ANOVA on the regression model to assess factor contributions

---
## Part 5: Non-Parametric Model (Neural Network)

Build a neural network regression model using **all** available features to predict SalePrice. This is also the model that produces your `submission.csv` for Kaggle scoring.

**Tasks:**
- Preprocess all features: handle missing values, encode categoricals, scale numerics
- Build and train a neural network (e.g. `sklearn.neural_network.MLPRegressor` or PyTorch)
- Evaluate on training data (RMSE, R²) and analyze residuals
- Generate predictions for the test set and save as `submission.csv`

**Important:** The Kaggle RMSE score is evaluated on the predictions from this model.